# Ingestion Pipeline Test

Test pipeline: Ingest **DOCX / PDF** → Normalize → Chunking → Embedding → ChromaDB


In [ ]:
# === Cell nhẹ: imports cơ bản + paths (chạy nhanh ~1s) ===
from pathlib import Path
import re, json, os
from datetime import datetime, timezone
from docx import Document
import pdfplumber
from dotenv import load_dotenv

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_DIR   = ROOT / 'data' / 'sample_pairs' / 'standalone'
PAIRS_DIR  = ROOT / 'data' / 'sample_pairs' / 'pairs'
CHROMA_DIR = ROOT / 'chroma_db'
OUTPUT_DIR = ROOT / 'outputs'
CHROMA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('ROOT       :', ROOT)
print('Standalone :', DATA_DIR)
print('Files      :', [f.name for f in sorted(DATA_DIR.glob('*.docx')) + sorted(DATA_DIR.glob('*.pdf'))])

: 

In [ ]:
# === Hàm đọc DOCX và PDF ===
def read_docx_text(path: Path) -> str:
    """Đọc file DOCX, trả về text."""
    doc = Document(path)
    return '\n'.join([p.text.strip() for p in doc.paragraphs if p.text.strip()])

def read_pdf_text(path: Path) -> str:
    """Đọc file PDF bằng pdfplumber, trả về text."""
    pages_text = []
    with pdfplumber.open(path) as pdf:
        for page in pdf.pages:
            text = page.extract_text()
            if text:
                pages_text.append(text.strip())
    return '\n'.join(pages_text)

def read_document(path: Path) -> str:
    """Đọc file DOCX hoặc PDF tự động theo đuôi file."""
    ext = path.suffix.lower()
    if ext == '.docx':
        return read_docx_text(path)
    elif ext == '.pdf':
        return read_pdf_text(path)
    else:
        raise ValueError(f'Không hỗ trợ định dạng: {ext}')

# Test đọc file đầu tiên
docs = sorted(DATA_DIR.glob('*.docx')) + sorted(DATA_DIR.glob('*.pdf'))
if docs:
    raw_text = read_document(docs[0])
    source_name = docs[0].name
else:
    source_name = 'inline_demo.docx'
    raw_text = 'Điều 1. Phạm vi\n1. Bên A thuê Bên B.\n\nĐiều 2. Thanh toán\n1. Thanh toán trong vòng 30 ngày.'

print(f'source = {source_name}')
print(f'type   = {Path(source_name).suffix}')
print(f'raw_len= {len(raw_text)}')

: 

In [ ]:
def normalize_text(text: str) -> str:
    text = text.replace('\xa0', ' ')
    text = re.sub(r'\r\n|\r', '\n', text)
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()

clean_text = normalize_text(raw_text)
print(clean_text[:300])

LUẬT
QUY HOẠCH
Căn cứ Hiến pháp nước Cộng hòa xã hội chủ nghĩa Việt Nam đã được sửa đổi, bổ sung một số điều theo Nghị quyết số 203/2025/QH15;
Quốc hội ban hành Luật Quy hoạch.
Chương I
QUY ĐỊNH CHUNG
Điều 1. Phạm vi điều chỉnh
Luật này quy định hệ thống quy hoạch; việc lập, thẩm định, quyết định ho


In [ ]:
# === Chunker đầy đủ: Header → Chương → Mục → Điều (chunk unit) + metadata Khoản/Điểm/Định nghĩa ===

CHUONG_RE     = re.compile(r'^Chương\s+([IVXLCDM\d]+)[\.:]?\s*(.*)', re.IGNORECASE)
MUC_RE        = re.compile(r'^Mục\s+(\d+)[\.:]?\s*(.*)', re.IGNORECASE)
DIEU_RE       = re.compile(r'^Điều\s+(\d+)[\.:]?\s*(.*)', re.IGNORECASE)
KHOAN_RE      = re.compile(r'^\d+\.', re.MULTILINE)
DIEM_RE       = re.compile(r'^[a-zđ]\)', re.MULTILINE | re.IGNORECASE)
DINH_NGHIA_RE = re.compile(r'giải thích từ ngữ|định nghĩa', re.IGNORECASE)

def chunk_full_hierarchy(text: str, doc_id: str, version: str) -> list:
    """
    Parse đầy đủ cấu trúc văn bản pháp luật Việt Nam:
      Header → Chương → Mục → Điều (đơn vị chunk)
    Args:
        text:    Văn bản đã normalize
        doc_id:  Tên file (không có đuôi), VD: '135_2025_QH15'
        version: VD: 'v1', 'v2' — không hard-code, truyền từ ngoài vào
    chunk_type: 'header' | 'article' | 'definition'
    """
    lines = text.split('\n')
    chunks = []
    chunk_idx = 1

    cur_chuong_so, cur_chuong_ten = '0', ''
    cur_muc_so,    cur_muc_ten    = '0', ''
    cur_dieu_no,   cur_dieu_ten   = None, ''
    cur_dieu_lines = []
    header_lines   = []
    in_header      = True

    def flush_dieu():
        nonlocal chunk_idx, cur_dieu_no, cur_dieu_ten, cur_dieu_lines
        if cur_dieu_no is None:
            return
        block = '\n'.join(cur_dieu_lines).strip()
        if not block:
            cur_dieu_no = None; cur_dieu_lines = []
            return
        khoan_count = len(KHOAN_RE.findall(block))
        diem_count  = len(DIEM_RE.findall(block))
        ctype = 'definition' if DINH_NGHIA_RE.search(cur_dieu_ten) else 'article'
        chunks.append({
            'chunk_id':       f'{doc_id}_{version}_{chunk_idx:04d}',
            'doc_id':         doc_id,
            'version':        version,
            'chunk_type':     ctype,
            'chuong_so':      cur_chuong_so,
            'chuong_ten':     cur_chuong_ten,
            'muc_so':         cur_muc_so,
            'muc_ten':        cur_muc_ten,
            'clause_id':      f'Dieu_{cur_dieu_no}',
            'article_number': cur_dieu_no,
            'article_title':  cur_dieu_ten,
            'khoan_count':    khoan_count,
            'diem_count':     diem_count,
            'text':           block,
            'char_len':       len(block),
            'created_at':     datetime.now(timezone.utc).isoformat()
        })
        chunk_idx += 1
        cur_dieu_no = None; cur_dieu_ten = ''; cur_dieu_lines = []

    for line in lines:
        stripped = line.strip()
        m_c = CHUONG_RE.match(stripped)
        m_m = MUC_RE.match(stripped)
        m_d = DIEU_RE.match(stripped)

        if m_c:
            flush_dieu()
            cur_chuong_so  = m_c.group(1)
            cur_chuong_ten = m_c.group(2).strip()
            cur_muc_so, cur_muc_ten = '0', ''
            in_header = False
        elif m_m:
            flush_dieu()
            cur_muc_so  = m_m.group(1)
            cur_muc_ten = m_m.group(2).strip()
            in_header = False
        elif m_d:
            flush_dieu()
            cur_dieu_no  = m_d.group(1)
            cur_dieu_ten = m_d.group(2).strip()
            cur_dieu_lines = [stripped]
            in_header = False
        else:
            if cur_dieu_no is not None:
                cur_dieu_lines.append(line)
            elif in_header and stripped:
                header_lines.append(line)

    flush_dieu()

    if header_lines:
        header_text = '\n'.join(header_lines).strip()
        if header_text:
            chunks.insert(0, {
                'chunk_id':       f'{doc_id}_{version}_0000',
                'doc_id':         doc_id,
                'version':        version,
                'chunk_type':     'header',
                'chuong_so':      '0',
                'chuong_ten':     '',
                'muc_so':         '0',
                'muc_ten':        '',
                'clause_id':      'header',
                'article_number': '0',
                'article_title':  '',
                'khoan_count':    0,
                'diem_count':     0,
                'text':           header_text,
                'char_len':       len(header_text),
                'created_at':     datetime.now(timezone.utc).isoformat()
            })

    return chunks

print('chunk_full_hierarchy() defined OK')

chunk_full_hierarchy() defined OK


In [ ]:
# === Ingest Test: Chạy pipeline + Kiểm thử tự động (không cần Embedding/Chroma) ===

TEST_VERSION = 'v1'   # ← Thay đây nếu muốn test 'v2', 'draft'...

test_results = []

for docx_path in sorted(DATA_DIR.glob('*.docx')) + sorted(DATA_DIR.glob('*.pdf')):
    doc_id = docx_path.stem
    print(f'\n{"="*60}')
    print(f'▶ {docx_path.name}')

    # --- Bước 1: Đọc file ---
    raw = read_document(docx_path)
    assert len(raw) > 100, f'[FAIL] {doc_id}: raw text quá ngắn ({len(raw)} ký tự)'
    print(f'  [OK] Đọc file: {len(raw)} ký tự')

    # --- Bước 2: Normalize ---
    clean = normalize_text(raw)
    assert '\xa0' not in clean,     f'[FAIL] {doc_id}: vẫn còn ký tự \\xa0 sau normalize'
    assert '\r' not in clean,       f'[FAIL] {doc_id}: vẫn còn ký tự \\r sau normalize'
    assert '   ' not in clean,      f'[FAIL] {doc_id}: vẫn còn khoảng trắng thừa sau normalize'
    print(f'  [OK] Normalize: {len(clean)} ký tự')

    # --- Bước 3: Chunking ---
    chunks = chunk_full_hierarchy(clean, doc_id=doc_id, version=TEST_VERSION)
    article_chunks = [c for c in chunks if c['chunk_type'] == 'article']
    def_chunks     = [c for c in chunks if c['chunk_type'] == 'definition']
    header_chunks  = [c for c in chunks if c['chunk_type'] == 'header']

    # Kiểm tra số lượng
    assert len(chunks) > 0,            f'[FAIL] {doc_id}: không có chunk nào'
    assert len(article_chunks) >= 10,  f'[FAIL] {doc_id}: chỉ có {len(article_chunks)} article chunk (cần ≥10)'
    assert len(header_chunks) == 1,    f'[FAIL] {doc_id}: header chunk = {len(header_chunks)} (cần đúng 1)'
    print(f'  [OK] Chunks: tổng={len(chunks)} (header={len(header_chunks)}, article={len(article_chunks)}, def={len(def_chunks)})')

    # Kiểm tra Điều 1
    dieu1 = next((c for c in chunks if c['article_number'] == '1'), None)
    assert dieu1 is not None,                          f'[FAIL] {doc_id}: không tìm thấy Điều 1'
    assert dieu1['text'].startswith('Điều 1'),         f'[FAIL] {doc_id}: Điều 1 không bắt đầu bằng "Điều 1"'
    assert dieu1['chunk_type'] in ('article','definition'), f'[FAIL] {doc_id}: Điều 1 có chunk_type lạ'
    print(f'  [OK] Điều 1: "{dieu1["article_title"]}" | Chương {dieu1["chuong_so"]}')

    # Kiểm tra không có chunk rỗng
    empty = [c for c in chunks if c['char_len'] <= 30]
    assert len(empty) == 0, f'[FAIL] {doc_id}: có {len(empty)} chunk quá ngắn (≤30 ký tự): {[c["chunk_id"] for c in empty]}'
    print(f'  [OK] Không có chunk rỗng (min char_len={min(c["char_len"] for c in chunks)})')

    # Kiểm tra chunk_id unique
    ids = [c['chunk_id'] for c in chunks]
    assert len(ids) == len(set(ids)), f'[FAIL] {doc_id}: chunk_id bị trùng!'
    print(f'  [OK] chunk_id unique: {len(ids)} IDs')

    # Kiểm tra metadata nhất quán
    assert all(c['doc_id'] == doc_id for c in chunks),         f'[FAIL] {doc_id}: doc_id không nhất quán'
    assert all(c['version'] == TEST_VERSION for c in chunks),  f'[FAIL] {doc_id}: version không nhất quán'
    assert all(c['chunk_type'] in ('header','article','definition') for c in chunks), \
        f'[FAIL] {doc_id}: chunk_type không hợp lệ'
    print(f'  [OK] Metadata nhất quán (doc_id, version, chunk_type)')

    # Kiểm tra Chương/Mục có được nhận ra
    chuongs = sorted(set(c['chuong_so'] for c in chunks if c['chuong_so'] != '0'))
    mucs    = sorted(set(c['muc_so']    for c in chunks if c['muc_so']    != '0'))
    assert len(chuongs) > 0, f'[FAIL] {doc_id}: không nhận ra Chương nào'
    print(f'  [OK] Cấu trúc: Chương={chuongs}, Mục={mucs}')

    test_results.append({'doc_id': doc_id, 'chunks': len(chunks), 'articles': len(article_chunks), 'passed': True})

print(f'\n{"="*60}')
print(f'✅ TẤT CẢ {len(test_results)} FILE ĐỀU PASS kiểm thử ingest!')


▶ 112_2025_QH15_586814.docx
  [OK] Đọc file: 107417 ký tự
  [OK] Normalize: 107417 ký tự
  [OK] Chunks: tổng=59 (header=1, article=57, def=1)
  [OK] Điều 1: "Phạm vi điều chỉnh" | Chương I
  [OK] Không có chunk rỗng (min char_len=176)
  [OK] chunk_id unique: 59 IDs
  [OK] Metadata nhất quán (doc_id, version, chunk_type)
  [OK] Cấu trúc: Chương=['I', 'II', 'III', 'IV', 'V', 'VI'], Mục=['1', '2']

▶ 114_2025_QH15_658530.docx
  [OK] Đọc file: 55952 ký tự
  [OK] Normalize: 55952 ký tự
  [OK] Chunks: tổng=47 (header=1, article=45, def=1)
  [OK] Điều 1: "Phạm vi điều chỉnh, đối tượng áp dụng" | Chương I
  [OK] Không có chunk rỗng (min char_len=100)
  [OK] chunk_id unique: 47 IDs
  [OK] Metadata nhất quán (doc_id, version, chunk_type)
  [OK] Cấu trúc: Chương=['I', 'II', 'III', 'IV', 'V', 'VI'], Mục=[]

▶ 116_2025_QH15_666020.docx
  [OK] Đọc file: 86385 ký tự
  [OK] Normalize: 86385 ký tự
  [OK] Chunks: tổng=46 (header=1, article=44, def=1)
  [OK] Điều 1: "Phạm vi điều chỉnh và đối tượng áp d

In [ ]:
# === Export chunks ra file JSONL (1 dòng = 1 chunk) theo doc_id + version ===
# JSONL tốt hơn JSON cho streaming/lazy load khi file lớn

exported_files = []

for docx_path in sorted(DATA_DIR.glob('*.docx')) + sorted(DATA_DIR.glob('*.pdf')):
    doc_id = docx_path.stem
    raw    = normalize_text(read_document(docx_path))
    chunks = chunk_full_hierarchy(raw, doc_id=doc_id, version=TEST_VERSION)

    out_path = OUTPUT_DIR / f'{doc_id}_{TEST_VERSION}_chunks.jsonl'
    with out_path.open('w', encoding='utf-8') as f:
        for c in chunks:
            f.write(json.dumps(c, ensure_ascii=False) + '\n')

    exported_files.append((out_path.name, len(chunks)))
    print(f'  Saved: {out_path.name}  ({len(chunks)} chunks)')

# Verify: đọc lại và kiểm tra
print()
for fname, expected_count in exported_files:
    fpath = OUTPUT_DIR / fname
    loaded = [json.loads(line) for line in fpath.read_text(encoding='utf-8').strip().split('\n')]
    assert len(loaded) == expected_count, f'[FAIL] {fname}: đọc lại được {len(loaded)} ≠ {expected_count}'
    assert all('chunk_id' in c and 'text' in c for c in loaded), f'[FAIL] {fname}: thiếu field'
    print(f'  [OK] {fname}: {len(loaded)} chunks đọc lại OK')

print(f'\n✅ Export JSONL hoàn tất: {len(exported_files)} file → outputs/')

  Saved: 112_2025_QH15_586814_v1_chunks.jsonl  (59 chunks)
  Saved: 114_2025_QH15_658530_v1_chunks.jsonl  (47 chunks)
  Saved: 116_2025_QH15_666020_v1_chunks.jsonl  (46 chunks)
  Saved: 127_2025_QH15_686325_v1_chunks.jsonl  (181 chunks)
  Saved: 133_2025_QH15_675211_v1_chunks.jsonl  (28 chunks)
  Saved: 135_2025_QH15_675213_v1_chunks.jsonl  (96 chunks)
  Saved: 143_2025_QH15_681550_v1_chunks.jsonl  (53 chunks)
  Saved: 148_2025_QH15_675262_v1_chunks.jsonl  (49 chunks)

  [OK] 112_2025_QH15_586814_v1_chunks.jsonl: 59 chunks đọc lại OK
  [OK] 114_2025_QH15_658530_v1_chunks.jsonl: 47 chunks đọc lại OK
  [OK] 116_2025_QH15_666020_v1_chunks.jsonl: 46 chunks đọc lại OK
  [OK] 127_2025_QH15_686325_v1_chunks.jsonl: 181 chunks đọc lại OK
  [OK] 133_2025_QH15_675211_v1_chunks.jsonl: 28 chunks đọc lại OK
  [OK] 135_2025_QH15_675213_v1_chunks.jsonl: 96 chunks đọc lại OK
  [OK] 143_2025_QH15_681550_v1_chunks.jsonl: 53 chunks đọc lại OK
  [OK] 148_2025_QH15_675262_v1_chunks.jsonl: 49 chunks đọc lại 

In [ ]:
# === Đăng nhập HuggingFace từ .env ===
from huggingface_hub import login

load_dotenv(ROOT / '.env')
hf_token = os.getenv('HF_TOKEN')

if not hf_token or hf_token == 'hf_your_token_here':
    raise ValueError("Chưa cấu hình HF_TOKEN trong file .env! Vào https://huggingface.co/settings/tokens để lấy token.")

login(token=hf_token)
print("HuggingFace login OK")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


HuggingFace login OK


In [ ]:
# === Cell nặng: Load embedding model + ChromaDB client (chạy ~20-30s lần đầu) ===
from sentence_transformers import SentenceTransformer
from pyvi import ViTokenizer
import chromadb

model_name = 'huyydangg/DEk21_hcmute_embedding'
embedder = SentenceTransformer(model_name, token=hf_token)

client = chromadb.PersistentClient(path=str(CHROMA_DIR))

# Test embed nhanh
texts_segmented = [ViTokenizer.tokenize(c['text']) for c in chunks]
vectors = embedder.encode(texts_segmented, convert_to_numpy=True, show_progress_bar=True)
print(f'vector_shape = {vectors.shape}')
print(f'ChromaDB client OK, path = {CHROMA_DIR}')

Batches: 100%|██████████| 2/2 [00:04<00:00,  2.25s/it]

vector_shape = (49, 768)
ChromaDB client OK, path = d:\PTIT\TTCS\PRJ\Rag-Based-Legal-Comparison\chroma_db


In [ ]:
# === Full pipeline: Ingest tất cả DOCX + PDF → Normalize → Chunk → Embed → ChromaDB ===

# Reset collection
try:
    client.delete_collection('legal_chunks')
except:
    pass
collection = client.get_or_create_collection('legal_chunks')

all_files = sorted(DATA_DIR.glob('*.docx')) + sorted(DATA_DIR.glob('*.pdf'))
print(f'Sẽ xử lý {len(all_files)} file: {[f.name for f in all_files]}\n')

all_chunks_export = []

for file_path in all_files:
    print(f'▶ {file_path.name}')

    raw = read_document(file_path)
    raw = normalize_text(raw)
    file_chunks = chunk_full_hierarchy(raw, doc_id=file_path.stem, version='v1')

    segs = [ViTokenizer.tokenize(c['text']) for c in file_chunks]
    vecs = embedder.encode(segs, convert_to_numpy=True, show_progress_bar=False)

    # Metadata: chỉ lưu kiểu str/int/float/bool (ChromaDB yêu cầu)
    collection.add(
        ids=[c['chunk_id'] for c in file_chunks],
        documents=[c['text'] for c in file_chunks],
        metadatas=[{
            'doc_id':         c['doc_id'],
            'version':        c['version'],
            'chunk_type':     c['chunk_type'],
            'chuong_so':      c['chuong_so'],
            'chuong_ten':     c['chuong_ten'],
            'muc_so':         c['muc_so'],
            'muc_ten':        c['muc_ten'],
            'clause_id':      c['clause_id'],
            'article_number': c['article_number'],
            'article_title':  c['article_title'],
            'khoan_count':    c['khoan_count'],
            'diem_count':     c['diem_count'],
        } for c in file_chunks],
        embeddings=[v.tolist() for v in vecs]
    )

    all_chunks_export.extend(file_chunks)

    n_h = sum(1 for c in file_chunks if c['chunk_type'] == 'header')
    n_a = sum(1 for c in file_chunks if c['chunk_type'] == 'article')
    n_d = sum(1 for c in file_chunks if c['chunk_type'] == 'definition')
    chuongs = sorted(set(c['chuong_so'] for c in file_chunks if c['chuong_so'] != '0'))
    print(f'  chunks={len(file_chunks)} (header={n_h}, article={n_a}, definition={n_d})'
          f'  chương={chuongs}  shape={vecs.shape}')

print(f'\nChroma total: {collection.count()} chunks')

out_path = OUTPUT_DIR / 'all_chunks_v1.json'
out_path.write_text(json.dumps(all_chunks_export, ensure_ascii=False, indent=2), encoding='utf-8')
print(f'Saved: {out_path}')

Sẽ xử lý 8 file: ['112_2025_QH15_586814.docx', '114_2025_QH15_658530.docx', '116_2025_QH15_666020.docx', '127_2025_QH15_686325.docx', '133_2025_QH15_675211.docx', '135_2025_QH15_675213.docx', '143_2025_QH15_681550.docx', '148_2025_QH15_675262.docx']

▶ 112_2025_QH15_586814.docx
  chunks=59 (header=1, article=57, definition=1)  chương=['I', 'II', 'III', 'IV', 'V', 'VI']  shape=(59, 768)
▶ 114_2025_QH15_658530.docx
  chunks=47 (header=1, article=45, definition=1)  chương=['I', 'II', 'III', 'IV', 'V', 'VI']  shape=(47, 768)
▶ 116_2025_QH15_666020.docx
  chunks=46 (header=1, article=44, definition=1)  chương=['I', 'II', 'III', 'IV', 'V', 'VI', 'VII', 'VIII']  shape=(46, 768)
▶ 127_2025_QH15_686325.docx
  chunks=181 (header=1, article=179, definition=1)  chương=['I', 'II', 'III', 'IV', 'IX', 'V', 'VI', 'VII', 'VIII', 'X', 'XI', 'XII', 'XIII', 'XIV', 'XV']  shape=(181, 768)
▶ 133_2025_QH15_675211.docx
  chunks=28 (header=1, article=26, definition=1)  chương=['I', 'II', 'III', 'IV', 'V', 'VI'

In [ ]:
# === Kiểm tra ChromaDB đang lưu gì ===
import chromadb

# Kết nối lại (an toàn nếu kernel restart)
_client     = chromadb.PersistentClient(path=str(CHROMA_DIR))
_collection = _client.get_or_create_collection('legal_chunks')

total = _collection.count()
print(f'Tổng chunks trong Chroma: {total}')
print(f'Collection name         : {_collection.name}')
print()

# --- 1. Thống kê theo doc_id ---
all_meta = _collection.get(include=['metadatas'])['metadatas']

from collections import Counter
doc_counts   = Counter(m['doc_id']       for m in all_meta)
type_counts  = Counter(m['chunk_type']   for m in all_meta)
chuong_dist  = Counter(m['chuong_so']    for m in all_meta)

print('── Số chunks theo doc_id ──')
for doc, cnt in sorted(doc_counts.items()):
    print(f'  {doc}: {cnt}')

print()
print('── Số chunks theo chunk_type ──')
for t, cnt in sorted(type_counts.items()):
    print(f'  {t}: {cnt}')

print()
print('── Phân bố Chương (top 10) ──')
for chuong, cnt in sorted(chuong_dist.items(), key=lambda x: -x[1])[:10]:
    label = f'Chương {chuong}' if chuong != '0' else 'header'
    print(f'  {label}: {cnt} chunks')

# --- 2. Kiểm tra metadata fields đầy đủ ---
print()
required_fields = ['doc_id','version','chunk_type','chuong_so','muc_so',
                   'clause_id','article_number','article_title','khoan_count','diem_count']
missing = [m for m in all_meta if not all(f in m for f in required_fields)]
if missing:
    print(f'⚠️  {len(missing)} chunk thiếu field metadata!')
else:
    print(f'[OK] Tất cả {total} chunks có đủ {len(required_fields)} metadata fields')

# --- 3. Xem 3 chunks mẫu (query bằng text) ---
print()
print('── 3 chunks mẫu (lấy từ đầu collection) ──')
sample = _collection.get(limit=3, include=['metadatas','documents'])
for i, (doc, meta) in enumerate(zip(sample['documents'], sample['metadatas'])):
    print(f'\n[Chunk {i+1}] {meta["clause_id"]} | {meta["chunk_type"]} | Chương {meta["chuong_so"]} | Mục {meta["muc_so"]}')
    print(f'  doc_id        : {meta["doc_id"]}')
    print(f'  article_title : {meta["article_title"]}')
    print(f'  khoan_count   : {meta["khoan_count"]}  |  diem_count: {meta["diem_count"]}')
    print(f'  text preview  : {doc[:120].strip()}...')

Tổng chunks trong Chroma: 559
Collection name         : legal_chunks

── Số chunks theo doc_id ──
  112_2025_QH15_586814: 59
  114_2025_QH15_658530: 47
  116_2025_QH15_666020: 46
  127_2025_QH15_686325: 181
  133_2025_QH15_675211: 28
  135_2025_QH15_675213: 96
  143_2025_QH15_681550: 53
  148_2025_QH15_675262: 49

── Số chunks theo chunk_type ──
  article: 543
  definition: 8
  header: 8

── Phân bố Chương (top 10) ──
  Chương II: 128 chunks
  Chương IV: 91 chunks
  Chương I: 90 chunks
  Chương V: 67 chunks
  Chương III: 49 chunks
  Chương VI: 40 chunks
  Chương VII: 23 chunks
  Chương XIII: 18 chunks
  Chương VIII: 12 chunks
  Chương IX: 9 chunks

[OK] Tất cả 559 chunks có đủ 10 metadata fields

── 3 chunks mẫu (lấy từ đầu collection) ──

[Chunk 1] header | header | Chương 0 | Mục 0
  doc_id        : 112_2025_QH15_586814
  article_title : 
  khoan_count   : 0  |  diem_count: 0
  text preview  : LUẬT
QUY HOẠCH
Căn cứ Hiến pháp nước Cộng hòa xã hội chủ nghĩa Việt Nam đã được sửa đổi, bổ

In [ ]:
# === Xem cụ thể 3 loại chunk đang lưu trong Chroma ===

for chunk_type in ['header', 'definition', 'article']:
    result = _collection.get(
        where={'chunk_type': chunk_type},
        limit=1,
        include=['metadatas', 'documents', 'embeddings']
    )
    meta = result['metadatas'][0]
    text = result['documents'][0]
    emb  = result['embeddings'][0]

    print(f'{"─"*65}')
    print(f'  chunk_type    : {meta["chunk_type"]}')
    print(f'  chunk_id      : {result["ids"][0]}')
    print(f'  doc_id        : {meta["doc_id"]}')
    print(f'  clause_id     : {meta["clause_id"]}')
    print(f'  article_title : {meta["article_title"] or "(không có)"}')
    print(f'  chuong_so     : {meta["chuong_so"]}  |  muc_so: {meta["muc_so"]}')
    print(f'  khoan_count   : {meta["khoan_count"]}  |  diem_count: {meta["diem_count"]}')
    print(f'  text ({len(text)} ký tự):')
    print(f'    {text[:200].strip()}')
    print(f'  embedding     : [{emb[0]:.4f}, {emb[1]:.4f}, ... {emb[-1]:.4f}]  (dim={len(emb)})')
    print()

─────────────────────────────────────────────────────────────────
  chunk_type    : header
  chunk_id      : 112_2025_QH15_586814_v1_0000
  doc_id        : 112_2025_QH15_586814
  clause_id     : header
  article_title : (không có)
  chuong_so     : 0  |  muc_so: 0
  khoan_count   : 0  |  diem_count: 0
  text (176 ký tự):
    LUẬT
QUY HOẠCH
Căn cứ Hiến pháp nước Cộng hòa xã hội chủ nghĩa Việt Nam đã được sửa đổi, bổ sung một số điều theo Nghị quyết số 203/2025/QH15;
Quốc hội ban hành Luật Quy hoạch.
  embedding     : [-0.1779, -0.1141, ... 0.4631]  (dim=768)

─────────────────────────────────────────────────────────────────
  chunk_type    : definition
  chunk_id      : 112_2025_QH15_586814_v1_0003
  doc_id        : 112_2025_QH15_586814
  clause_id     : Dieu_3
  article_title : Giải thích từ ngữ
  chuong_so     : I  |  muc_so: 0
  khoan_count   : 22  |  diem_count: 0
  text (6280 ký tự):
    Điều 3. Giải thích từ ngữ
Trong Luật này, các từ ngữ dưới đây được hiểu như sau:
1. Quy hoạch l